# Holdout (run once, only after Stage 3 passes and config.yaml is frozen)
1. Set `status.frozen: true` in config.yaml and log the commit in RUNLOG.md.
2. In a shell: `GAMMA_EDGE_RUN_HOLDOUT=1 python -m src.robustness holdout-prep`
3. Price and pull holdout trades: `GAMMA_EDGE_RUN_HOLDOUT=1 python -m src.ingest_futures trades --holdout --price-only`, then `--approve-usd X`
4. Run the cell below with the in-sample confirmed expectancy.

Pass: same sign and at least half the in-sample expectancy.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # run from notebooks/; src/ lives one level up
os.chdir("..")
%matplotlib inline
from src import analysis, store, plots
from src.config import load_config
cfg = load_config()

In [ ]:
import os
IN_SAMPLE_EXPECTANCY = None   # from 03_flow
CARRY = ['gamma_only', 'both', 'structural_only']
if IN_SAMPLE_EXPECTANCY is not None and os.environ.get('GAMMA_EDGE_RUN_HOLDOUT') == '1':
    from src import robustness
    out = robustness.holdout(IN_SAMPLE_EXPECTANCY, CARRY)
    print(analysis.to_json({k: out[k] for k in ('holdout_expectancy_r','in_sample_expectancy_r','verdict_vs_rules')}))
else:
    print('Holdout sealed.')

## Robustness (in-sample, report only)
Expectancy must stay positive in >= 80% of one-notch nudges.

In [ ]:
from src import robustness
nd = robustness.nudges(cfg, CARRY)
print(nd.attrs)
nd